In [ ]:
####### Link to presentation ####################

In [ ]:
# https://canva.link/zgwudrx3p9z1lia

In [ ]:
################# Libraries, version check ###########################

In [ ]:
import pandas as pd

In [ ]:
import sys
print(sys.executable)

In [ ]:
import geopandas as gpd

print(gpd.__version__)

In [ ]:
import fiona

print(fiona.__version__)

In [ ]:
import pyogrio

print(pyogrio.__version__)

In [ ]:
############## pois dataset (points of interest) ############

In [ ]:
import geopandas as gpd

pois = gpd.read_file(
    r"C:\Users\jforo\OneDrive\Desktop\Ironhack\Projects\project_3\data\Geofabrik\gis_osm_pois_free_1.shp"
)

In [ ]:
pois.head()

In [ ]:
pois.columns

In [ ]:
pois.shape



In [ ]:
pois["fclass"].value_counts()

In [ ]:
pois["fclass"].value_counts().head(50)

In [ ]:
##### extra code to see all files in the dataset ####

In [ ]:
from pathlib import Path

folder = Path(r"C:\Users\jforo\OneDrive\Desktop\Ironhack\Projects\project_3\data\Geofabrik")

for file in sorted(folder.glob("*.shp")):
    print(file.name)

In [ ]:
##### admin dataset: to define Amsterdam boudaries #####

## Filter POIs to Amsterdam

The original POI dataset covers the entire province of Noord-Holland.

To focus the analysis on Amsterdam, we extracted the municipality boundary
(admin_level8) and used it to clip the POI dataset.

The resulting dataset contains only points of interest located within the
municipality of Amsterdam.

In [ ]:
admin = gpd.read_file(
    r"C:\Users\jforo\OneDrive\Desktop\Ironhack\Projects\project_3\data\Geofabrik\gis_osm_adminareas_a_free_1.shp"
)

In [ ]:
admin.head()


In [ ]:
admin.columns

In [ ]:
admin.shape

In [ ]:
admin[admin["name"].str.contains("Amsterdam", case=False, na=False)]

In [ ]:
amsterdam = admin[
    (admin["name"] == "Amsterdam") &
    (admin["fclass"] == "admin_level8")
]

In [ ]:
amsterdam.shape

In [ ]:
amsterdam_pois = gpd.clip(pois, amsterdam)

In [ ]:
amsterdam_pois.shape

In [ ]:
amsterdam_pois["fclass"].value_counts().head(50)


In [ ]:
amsterdam_pois.head()

In [ ]:
pc4 = gpd.read_file(
    r"C:\Users\jforo\OneDrive\Desktop\Ironhack\Projects\project_3\data\Geofabrik\cbs_pc4_2024.gpkg"
)

In [ ]:
pc4.head()


In [ ]:
pc4.columns

In [ ]:
pc4.shape

In [ ]:
[col for col in pc4.columns if "gemeente" in col.lower()]

In [ ]:
[col for col in pc4.columns if "amsterdam" in col.lower()]

In [ ]:
print(amsterdam.crs)
print(pc4.crs)

In [ ]:
# to convert both datasets to the same EPSG
amsterdam = amsterdam.to_crs(epsg=28992)

In [ ]:
print(amsterdam.crs)
print(pc4.crs)

In [ ]:
# postcodes in Amsterdam
amsterdam_pc4 = gpd.sjoin(
    pc4,
    amsterdam,
    how="inner",
    predicate="intersects"
)

In [ ]:
amsterdam_pc4.shape

In [ ]:
amsterdam_pc4[["postcode"]].head()

In [ ]:
amsterdam_pc4["postcode"].nunique()

In [ ]:
amsterdam_pc4["postcode"].duplicated().sum()

In [ ]:
amsterdam_pc4.columns

In [ ]:
amsterdam_pc4 = amsterdam_pc4[["postcode", "geometry"]].copy()

In [ ]:
amsterdam_pc4.head(5)

In [ ]:
analysis_df = amsterdam_pc4.copy()

In [ ]:
# Make sure both GeoDataFrames use the same CRS
amsterdam_pois = amsterdam_pois.to_crs(amsterdam_pc4.crs)

# Spatial join: assign each POI to the postal code polygon it falls within
pois_pc4 = gpd.sjoin(
    amsterdam_pois,
    amsterdam_pc4,
    how="inner",
    predicate="within"
)

# Keep only the relevant columns
pois_pc4 = pois_pc4[["postcode", "fclass", "name", "geometry","code"]].copy()

# Inspect the result
print(pois_pc4.shape)
pois_pc4.head()

In [ ]:
pois_pc4.shape

In [ ]:
pois_pc4["postcode"].nunique()

In [ ]:
pois_pc4["fclass"].value_counts()

In [ ]:
pois_pc4.to_csv(
    r"C:\Users\jforo\OneDrive\Desktop\Ironhack\Projects\project_3\data\pois_pc4.csv",
    index=False
)

In [ ]:
# example of "code" (2205 = playground)
pois_pc4[pois_pc4["code"] == 2205].value_counts()

In [ ]:
pois_pc4.head(5)

In [ ]:
####### code to add transport data ###############

In [ ]:
transport = gpd.read_file(
    r"C:\Users\jforo\OneDrive\Desktop\Ironhack\Projects\project_3\data\Geofabrik\gis_osm_transport_a_free_1.shp"
)

In [ ]:
transport.head(5)

In [ ]:
transport['fclass'].value_counts()

In [ ]:
transport.shape

In [ ]:
transport = transport.to_crs(amsterdam_pc4.crs)

In [ ]:
transport = gpd.clip(transport, amsterdam)

In [ ]:
transport_pc4 = gpd.sjoin(
    transport,
    amsterdam_pc4,
    how="inner",
    predicate="within"
)

In [ ]:
transport_pc4.head(5)

In [ ]:
transport_pc4.shape

In [ ]:
transport_pc4 = transport_pc4[
    ["postcode", "code", "fclass", "name", "geometry"]
].copy()

In [ ]:
pois_pc4["source"] = "POI"
transport_pc4["source"] = "Transport"

In [ ]:
#### master dataset ####

In [ ]:
# master_df is a GeoDataFrame (column 'geometry')
master_df = pd.concat(
    [pois_pc4, transport_pc4],
    ignore_index=True
)

In [ ]:
master_df.shape

In [ ]:
pois_pc4.shape

In [ ]:
master_df['name'].value_counts()

In [ ]:
master_df.head()
master_df.shape
master_df["postcode"].unique()

In [ ]:
master_df[master_df["source"] == "Transport"]

In [ ]:
master_df[master_df["code"] == 2033]

In [ ]:
master_df.to_csv(
    r"C:\Users\jforo\OneDrive\Desktop\Ironhack\Projects\project_3\data\master_df.csv",
    index=False
)

In [ ]:
master_df["fclass"].nunique()

In [ ]:
master_df["code"].value_counts().head(50)

In [ ]:
master_df.info()

In [ ]:
master_df.isnull().sum()

In [ ]:
### railways: exploration ####

In [ ]:
railways = gpd.read_file(
    r"C:\Users\jforo\OneDrive\Desktop\Ironhack\Projects\project_3\data\Geofabrik\gis_osm_railways_free_1.shp"
)

In [ ]:
railways.head(5)

In [ ]:
railways.shape

In [ ]:
# only 2 bus stops in master_df
master_df[master_df["fclass"].str.contains("stop", case=False, na=False)]

In [ ]:
# only 6 transport stations in master_df
master_df[master_df["fclass"].str.contains("station", case=False, na=False)]

In [ ]:
master_df[master_df["postcode"] == 1151]

In [ ]:
###############  data cleaning, aggregation ########################

In [ ]:
master_df["code_group"] = master_df["code"].astype(str).str[:2]

In [ ]:
master_df["postcode"].value_counts()

In [ ]:
codes_to_keep = [2101, 2110, 2111, 2120, 2121, 2129, 2201, 2203, 2204, 2205, 2206, 2251, 2252, 2253, 2254, 2501, 2502, 2516, 2511, 2528]

In [ ]:
filtered_df = master_df[master_df["code"].isin(codes_to_keep)]

In [ ]:
filtered_df["code_group"].value_counts()

In [ ]:
filtered_df.shape

In [ ]:
filtered_df.head()

In [ ]:
filtered_df["postcode"].value_counts()

In [ ]:
summary_df = filtered_df.pivot_table(
    index="postcode",
    columns="code_group",
    values="code",
    aggfunc="count",
    fill_value=0
).reset_index()

In [ ]:
summary_df.head(5)

In [ ]:
summary_df = summary_df.rename(columns={
    '21': "health_count",
    '22': "leisure_count",
    '25': "shops_count",
})

In [ ]:
summary_df.head(5)

In [ ]:
house_units_df = pd.read_csv(r"C:\Users\jforo\OneDrive\Desktop\Ironhack\Projects\project_3\data\house_units.csv")

In [ ]:
house_units_df = house_units_df.rename(columns={
    'postNumber': "postcode",
    'count':'house_units_count'
})

In [ ]:
house_units_df.head()

In [ ]:
summary_df = summary_df.merge(
    house_units_df,
    on="postcode",
    how="left"
)

In [ ]:
summary_df["house_units_count"] = summary_df["house_units_count"].fillna(0)

In [ ]:
summary_df.head()

In [ ]:
summary_df.isnull().sum()

In [ ]:
summary_df["postcode"].duplicated().sum()

In [ ]:
summary_df.to_csv(
    r"C:\Users\jforo\OneDrive\Desktop\Ironhack\Projects\project_3\data\summary_df.csv",
    index=False
)

In [ ]:
prices_df = pd.read_csv(r"C:\Users\jforo\OneDrive\Desktop\Ironhack\Projects\project_3\data\prices_df.csv")

In [ ]:
prices_df.head(5)

In [ ]:
summary_df = summary_df.merge(prices_df, on="postcode", how="left")

In [ ]:
summary_df

In [ ]:
summary_df.columns = summary_df.columns.str.lower()

In [ ]:
summary_df.isnull().sum()

In [ ]:
count_columns = [
    "health_count",
    "leisure_count",
    "shops_count"
]

for col in count_columns:
    summary_df[f"{col}_per_1000"] = (
        summary_df[col] * 1000 / summary_df["house_units_count"]
    ).round(2)

In [ ]:
summary_df

In [ ]:
###################### visualization  ########################

In [ ]:
#### 1 - Maps #############

In [ ]:
map_df = summary_df.merge(
    amsterdam_pc4,
    on="postcode",
    how="left"
)

In [ ]:
map_df

map_df.to_csv(
    r"C:\Users\jforo\OneDrive\Desktop\Ironhack\Projects\project_3\data\map_df.csv",
    index=False
)

In [ ]:
import geopandas as gpd

map_df = gpd.GeoDataFrame(
    map_df,
    geometry="geometry",
    crs=amsterdam_pc4.crs
)

In [ ]:
print(summary_df.shape)
print(map_df.shape)
print(amsterdam_pc4.shape)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
def plot_choropleth(gdf, column, title, cmap):
    fig, ax = plt.subplots(figsize=(10, 10))

    gdf.plot(
        column=column,
        cmap=cmap,
        linewidth=0.5,
        edgecolor="black",
        legend=True,
        missing_kwds={
            "color": "lightgrey",
            "label": "No data"
        },
        ax=ax
    )

    ax.set_title(title, fontsize=16, fontweight="bold")
    ax.axis("off")

    plt.show()

In [ ]:
plot_choropleth(
    map_df,
    column="health_count_per_1000",
    title="Health Amenities by Postcode",
    cmap="Reds"
)

In [ ]:
plot_choropleth(
    map_df,
    column="leisure_count_per_1000",
    title="Leisure Amenities by Postcode",
    cmap="Greens"
)

In [ ]:
plot_choropleth(
    map_df,
    column="shops_count_per_1000",
    title="Shops by Postcode",
    cmap="Blues"
)

In [ ]:
plot_choropleth(
    map_df,
    column="house_units_count",
    title="Housing Units by Postcode",
    cmap="Purples"
)

In [ ]:
plot_choropleth(
    map_df,
    column="price",
    title="Average House Price by Postcode",
    cmap="OrRd"
)

In [ ]:
import matplotlib.pyplot as plt

# Copy dataframe
map_plot = map_df.copy()

# Default color
map_plot["color"] = "#EEF4FA"   # light blue

# Assign colors
map_plot.loc[map_plot["postcode"].isin([1381, 1384, 1082]), "color"] = "#D73027"  # Health
map_plot.loc[map_plot["postcode"].isin([1026, 1027, 1383]), "color"] = "#31A354"  # Leisure
map_plot.loc[map_plot["postcode"].isin([1012, 1093, 1047]), "color"] = "#3182BD"  # Shops
map_plot.loc[map_plot["postcode"].isin([1102, 1018, 1013]), "color"] = "#756BB1"  # House Units
map_plot.loc[map_plot["postcode"].isin([1071, 1016, 1015]), "color"] = "#F16913"  # Price

# Plot
fig, ax = plt.subplots(figsize=(12,12))

map_plot.plot(
    ax=ax,
    color=map_plot["color"],
    edgecolor="black",
    linewidth=0.5
)

# Labels

for _, row in map_plot.iterrows():
    if row.geometry.area > 1000000:
        centroid = row.geometry.centroid
        ax.text(
            centroid.x,
            centroid.y,
            str(row["postcode"]),
            fontsize=8,
            ha="center",
            va="center"
        )

ax.set_title("Amsterdam PC4 Postal Codes - Top 3 by Category",
             fontsize=16,
             fontweight="bold")

ax.axis("off")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Copy dataframe
map_plot = map_df.copy()

# Default color
map_plot["color"] = "#EEF4FA"   # light blue

# Assign colors
map_plot.loc[map_plot["postcode"].isin([1381, 1384, 1082, 1023, 1086]), "color"] = "#D73027"  # Health
map_plot.loc[map_plot["postcode"].isin([1026, 1027, 1383, 1109, 1108]), "color"] = "#31A354"  # Leisure
map_plot.loc[map_plot["postcode"].isin([1012, 1093, 1047, 1094, 1072]), "color"] = "#3182BD"  # Shops
map_plot.loc[map_plot["postcode"].isin([1102, 1018, 1013, 1019, 1055]), "color"] = "#756BB1"  # House Units
map_plot.loc[map_plot["postcode"].isin([1071, 1016, 1015, 1017, 1012]), "color"] = "#F16913"  # Price

# Plot
fig, ax = plt.subplots(figsize=(12,12))

map_plot.plot(
    ax=ax,
    color=map_plot["color"],
    edgecolor="black",
    linewidth=0.5
)

# Labels

for _, row in map_plot.iterrows():
    if row.geometry.area > 900000:
        centroid = row.geometry.centroid
        ax.text(
            centroid.x,
            centroid.y,
            str(row["postcode"]),
            fontsize=8,
            ha="center",
            va="center"
        )

ax.set_title("Amsterdam PC4 Postal Codes - Top 5 by Category",
             fontsize=16,
             fontweight="bold")

ax.axis("off")

plt.show()

In [ ]:
summary_df.describe().round(2)

In [ ]:
######  2 - Bar charts ##############

In [ ]:
import matplotlib.pyplot as plt

def top10_barh(df, column, title, color):

    top10 = (
        df.sort_values(column, ascending=False)
          .head(5)
          .sort_values(column)
    )

    plt.figure(figsize=(6, 4))   # Smaller figure

    plt.barh(
        top10["postcode"].astype(str),
        top10[column],
        color=color,
        height=0.6
    )

    plt.title(title, fontsize=16, fontweight="bold")
    plt.xlabel(column.replace("_", " ").title(), fontsize=13)
    plt.ylabel("Postcode", fontsize=13)

    # Larger tick labels
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    plt.grid(axis="x", alpha=0.3)

    # Value labels
    for i, value in enumerate(top10[column]):
        plt.text(
            value,
            i,
            f"{value:,.2f}",
            va="center",
            ha="left",
            fontsize=12,
            fontweight="bold"
        )

    plt.tight_layout()
    plt.show()

In [ ]:
def top10_barh_sd(df, column, title, color):

    top10 = (
        df.sort_values(column, ascending=False)
          .head(5)
          .sort_values(column)
    )

    plt.figure(figsize=(6, 4))   # Smaller figure

    plt.barh(
        top10["postcode"].astype(str),
        top10[column],
        color=color,
        height=0.6
    )

    plt.title(title, fontsize=16, fontweight="bold")
    plt.xlabel(column.replace("_", " ").title(), fontsize=13)
    plt.ylabel("Postcode", fontsize=13)

    # Larger tick labels
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    plt.grid(axis="x", alpha=0.3)

    # Value labels
    for i, value in enumerate(top10[column]):
        plt.text(
            value,
            i,
            f"{value:,.0f}",
            va="center",
            ha="left",
            fontsize=12,
            fontweight="bold"
        )

    plt.tight_layout()
    plt.show()



In [ ]:
top10_barh(
    summary_df,
    "health_count_per_1000",
    "Top 5 Postcodes by Health Amenities",
    "tomato"
)

In [ ]:
top10_barh(
    summary_df,
    "leisure_count_per_1000",
    "Top 5 Postcodes by Leisure Amenities",
    "forestgreen"
)

In [ ]:
top10_barh(
    summary_df,
    "shops_count_per_1000",
    "Top 5 Postcodes by Shops",
    "steelblue"
)

In [ ]:
top10_barh_sd(
    summary_df,
    "price",
    "Top 5 Postcodes by House Price",
    "darkorange"
)

In [ ]:
top10_barh_sd(
    summary_df,
    "house_units_count",
    "Top 10 Postcodes by House Units",
    "mediumpurple"
)

In [ ]:
################## heatmap ###############################

In [ ]:
heatmap_df = summary_df[[
    "health_count_per_1000",
    "leisure_count_per_1000",
    "shops_count_per_1000",
    "price"
]]

In [ ]:
corr = heatmap_df.corr(method="pearson")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="PRGn",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Correlation"}
)

plt.title("Correlation between Amenities and House Prices", fontsize=15)

plt.tight_layout()

plt.show()

In [ ]:
####### correlations, scatter plots #########

In [ ]:
sns.pairplot(heatmap_df)  # plot pairwise relationships in dataset  # useful for exploratory analysis
plt.show()  # display plot

In [ ]:
correlation = heatmap_df.corr(numeric_only=True)  # compute correlation matrix for numeric columns
sns.heatmap(correlation, annot=True)  # plot heatmap with annotations
plt.title("Correlation beetwen Amenities and House Prices")  # add title
plt.show()  # display plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 7))

# Scatter plots
plt.scatter(summary_df["health_count_per_1000"],
            summary_df["price"],
            color="red",
            alpha=0.7,
            label="Health")

plt.scatter(summary_df["leisure_count_per_1000"],
            summary_df["price"],
            color="green",
            alpha=0.7,
            label="Leisure")

plt.scatter(summary_df["shops_count_per_1000"],
            summary_df["price"],
            color="blue",
            alpha=0.7,
            label="Shops")


plt.xlabel("Variable value")
plt.ylabel("Average House Price (€)")
plt.title("Relationship between House Price and Amenities")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Create a 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# ---------------- Health ----------------
axes[0, 0].scatter(
    summary_df["health_count_per_1000"],
    summary_df["price"],
    color="red",
    alpha=0.7
)
axes[0, 0].set_title("Price vs Health")
axes[0, 0].set_xlabel("Health amenities per 1,000 house units")
axes[0, 0].set_ylabel("Average House Price (€)")
axes[0, 0].grid(alpha=0.3)

# ---------------- Leisure ----------------
axes[0, 1].scatter(
    summary_df["leisure_count_per_1000"],
    summary_df["price"],
    color="green",
    alpha=0.7
)
axes[0, 1].set_title("Price vs Leisure")
axes[0, 1].set_xlabel("Leisure amenities per 1,000 house units")
axes[0, 1].set_ylabel("Average House Price (€)")
axes[0, 1].grid(alpha=0.3)

# ---------------- Shops ----------------
axes[1, 0].scatter(
    summary_df["shops_count_per_1000"],
    summary_df["price"],
    color="blue",
    alpha=0.7
)
axes[1, 0].set_title("Price vs Shops")
axes[1, 0].set_xlabel("Shops per 1,000 house units")
axes[1, 0].set_ylabel("Average House Price (€)")
axes[1, 0].grid(alpha=0.3)

# ---------------- House Units ----------------
axes[1, 1].scatter(
    summary_df["house_units_count"],
    summary_df["price"],
    color="purple",
    alpha=0.7
)
axes[1, 1].set_title("Price vs House Units")
axes[1, 1].set_xlabel("House Units")
axes[1, 1].set_ylabel("Average House Price (€)")
axes[1, 1].grid(alpha=0.3)


axes[0, 0].set_xlim(0, 5)   # Health
axes[0, 1].set_xlim(0, 5)   # Leisure
axes[1, 0].set_xlim(0, 5)   # Shops


# Improve spacing
plt.suptitle("Relationship between House Prices and Urban Variables",
             fontsize=16,
             fontweight="bold")

plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# -------- Health --------
sns.regplot(
    data=summary_df,
    x="health_count_per_1000",
    y="price",
    ax=axes[0, 0],
    scatter_kws={"color": "red", "alpha": 0.7},
    line_kws={"color": "black", "linewidth": 2}
)
axes[0, 0].set_title("Price vs Health")
axes[0, 0].set_xlabel("Health amenities per 1,000 house units")
axes[0, 0].set_ylabel("Average House Price (€)")
axes[0, 0].grid(alpha=0.3)

# -------- Leisure --------
sns.regplot(
    data=summary_df,
    x="leisure_count_per_1000",
    y="price",
    ax=axes[0, 1],
    scatter_kws={"color": "green", "alpha": 0.7},
    line_kws={"color": "black", "linewidth": 2}
)
axes[0, 1].set_title("Price vs Leisure")
axes[0, 1].set_xlabel("Leisure amenities per 1,000 house units")
axes[0, 1].set_ylabel("Average House Price (€)")
axes[0, 1].grid(alpha=0.3)

# -------- Shops --------
sns.regplot(
    data=summary_df,
    x="shops_count_per_1000",
    y="price",
    ax=axes[1, 0],
    scatter_kws={"color": "blue", "alpha": 0.7},
    line_kws={"color": "black", "linewidth": 2}
)
axes[1, 0].set_title("Price vs Shops")
axes[1, 0].set_xlabel("Shops per 1,000 house units")
axes[1, 0].set_ylabel("Average House Price (€)")
axes[1, 0].grid(alpha=0.3)

# -------- House Units --------
sns.regplot(
    data=summary_df,
    x="house_units_count",
    y="price",
    ax=axes[1, 1],
    scatter_kws={"color": "purple", "alpha": 0.7},
    line_kws={"color": "black", "linewidth": 2}
)
axes[1, 1].set_title("Price vs House Units")
axes[1, 1].set_xlabel("House Units")
axes[1, 1].set_ylabel("Average House Price (€)")
axes[1, 1].grid(alpha=0.3)

# Figure title
plt.suptitle(
    "Relationship between House Prices and Urban Variables",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()